In [1]:
# ── CELL 1: Imports and Initialisation ──────────────────────────────────────
import ee
import geemap
import os

ee.Initialize(project='ee-festac')

# Load Amuwo Odofin boundary using stringContains to handle GAUL spacing quirk
amuwo_odofin = ee.FeatureCollection("FAO/GAUL/2015/level2") \
                 .filter(ee.Filter.eq('ADM0_NAME', 'Nigeria')) \
                 .filter(ee.Filter.eq('ADM1_NAME', 'Lagos')) \
                 .filter(ee.Filter.stringContains('ADM2_NAME', 'Amuwo Odofin'))

# Verify boundary loaded correctly before extracting geometry
feature_count = amuwo_odofin.size().getInfo()
if feature_count == 0:
    raise ValueError("Boundary returned 0 features — check ADM2_NAME filter")

amuwo_geom = amuwo_odofin.geometry()

print("✓ GEE initialised")
print("✓ Amuwo Odofin boundary loaded and verified")
print(f"  Feature count: {feature_count}")

/home/deysholey/.local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


✓ GEE initialised
✓ Amuwo Odofin boundary loaded and verified
  Feature count: 1


In [2]:
# ── CELL 2: CHIRPS Rainfall Acquisition ─────────────────────────────────────
# Source: Climate Hazards Group InfraRed Precipitation with Station data
# Resolution: ~5km | Temporal coverage: 1981-present
# GEE ID: UCSB-CHG/CHIRPS/DAILY
#
# Strategy: compute mean annual rainfall and rainfall intensity metrics
# over a 10-year period (2013-2023) to capture climate variability
# while remaining computationally tractable for a pilot study

# Define analysis period
start_year = 2013
end_year = 2023

chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY") \
           .filter(ee.Filter.date(f'{start_year}-01-01', f'{end_year}-12-31')) \
           .filterBounds(amuwo_geom)

# Mean annual rainfall (mm/year)
# Sum daily rainfall per year then average across years
annual_rainfall = chirps.sum() \
                        .divide(end_year - start_year) \
                        .clip(amuwo_geom) \
                        .rename('mean_annual_rainfall')

# Rainfall intensity: mean of rainy days only (days with > 1mm precipitation)
# This distinguishes heavy episodic rain from light drizzle
rainy_days = chirps.map(lambda img: img.gt(1)) \
                   .sum() \
                   .divide(end_year - start_year) \
                   .clip(amuwo_geom) \
                   .rename('mean_rainy_days_per_year')

# Extreme rainfall frequency: days exceeding 50mm (heavy rainfall threshold)
# 50mm/day is widely used in West African flood literature as flood-triggering threshold
extreme_rain = chirps.map(lambda img: img.gt(50)) \
                     .sum() \
                     .divide(end_year - start_year) \
                     .clip(amuwo_geom) \
                     .rename('extreme_rain_frequency')

# Verify statistics
rainfall_stats = annual_rainfall.reduceRegion(
    reducer=ee.Reducer.minMax().combine(
        reducer2=ee.Reducer.mean(),
        sharedInputs=True
    ),
    geometry=amuwo_geom,
    scale=5000,
    maxPixels=1e9
).getInfo()

print("✓ CHIRPS rainfall acquired (2013-2023)")
print()
print("Mean Annual Rainfall Statistics (mm/year):")
print(f"  Minimum : {rainfall_stats['mean_annual_rainfall_min']:.2f} mm")
print(f"  Maximum : {rainfall_stats['mean_annual_rainfall_max']:.2f} mm")
print(f"  Mean    : {rainfall_stats['mean_annual_rainfall_mean']:.2f} mm")

✓ CHIRPS rainfall acquired (2013-2023)

Mean Annual Rainfall Statistics (mm/year):
  Minimum : 1509.21 mm
  Maximum : 1885.31 mm
  Mean    : 1662.90 mm


In [3]:
# ── CELL 3: ESA WorldCover Land Use / Land Cover ─────────────────────────────
# Source: European Space Agency WorldCover 2021
# Resolution: 10 metres — highest resolution global LULC freely available
# GEE ID: ESA/WorldCover/v200
#
# WorldCover class values:
# 10 = Tree cover       | 20 = Shrubland      | 30 = Grassland
# 40 = Cropland         | 50 = Built-up       | 60 = Bare/sparse vegetation
# 70 = Snow and ice     | 80 = Permanent water| 90 = Herbaceous wetland
# 95 = Mangroves        | 100 = Moss and lichen

lulc = ee.ImageCollection("ESA/WorldCover/v200") \
         .first() \
         .clip(amuwo_geom) \
         .rename('lulc')

# Compute percentage coverage of key flood-relevant classes
# Built-up (50): impervious surfaces amplify runoff
# Permanent water (80): baseline water bodies
# Herbaceous wetland (90): natural flood retention areas
# Mangroves (95): coastal flood buffer zones

def compute_class_percentage(image, class_value, class_name, geometry, scale=10):
    total_pixels = image.unmask(0).reduceRegion(
        reducer=ee.Reducer.count(),
        geometry=geometry,
        scale=scale,
        maxPixels=1e9
    ).get('lulc')
    
    class_pixels = image.eq(class_value).selfMask().reduceRegion(
        reducer=ee.Reducer.count(),
        geometry=geometry,
        scale=scale,
        maxPixels=1e9
    ).get('lulc')
    
    total = ee.Number(total_pixels)
    class_count = ee.Number(class_pixels).divide(total).multiply(100)
    return class_count.getInfo()

print("✓ ESA WorldCover 2021 loaded")
print()
print("Land Cover Distribution in Amuwo Odofin LGA:")
print(f"  Built-up surfaces    : {compute_class_percentage(lulc, 50, 'Built-up', amuwo_geom):.2f}%")
print(f"  Permanent water      : {compute_class_percentage(lulc, 80, 'Water', amuwo_geom):.2f}%")
print(f"  Herbaceous wetland   : {compute_class_percentage(lulc, 90, 'Wetland', amuwo_geom):.2f}%")
print(f"  Tree cover           : {compute_class_percentage(lulc, 10, 'Tree cover', amuwo_geom):.2f}%")
print(f"  Mangroves            : {compute_class_percentage(lulc, 95, 'Mangroves', amuwo_geom):.2f}%")

✓ ESA WorldCover 2021 loaded

Land Cover Distribution in Amuwo Odofin LGA:
  Built-up surfaces    : 40.60%
  Permanent water      : 12.74%
  Herbaceous wetland   : 0.76%
  Tree cover           : 16.33%
  Mangroves            : 14.61%


In [4]:
# ── CELL 4: Sentinel-2 NDVI Acquisition ─────────────────────────────────────
# Source: ESA Copernicus Sentinel-2 Surface Reflectance (Harmonised)
# Resolution: 10 metres
# GEE ID: COPERNICUS/S2_SR_HARMONIZED
#
# NDVI = (NIR - Red) / (NIR + Red)
# Band mapping for Sentinel-2: NIR = B8, Red = B4
# Value range: -1 to +1
# High NDVI (>0.5) = dense vegetation = pervious surface = lower runoff
# Low NDVI (<0.2) = sparse/no vegetation = impervious or bare = higher runoff
#
# Cloud masking is applied using the SCL (Scene Classification Layer)
# to ensure only clear-sky pixels contribute to the composite

def mask_s2_clouds(image):
    # SCL classes to mask: 3=cloud shadow, 8=medium cloud, 9=high cloud, 10=thin cirrus
    scl = image.select('SCL')
    cloud_mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return image.updateMask(cloud_mask)

def compute_ndvi(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('ndvi')
    return image.addBands(ndvi)

# Dry season composite (November-March) to minimise cloud cover
# Lagos dry season aligns with Harmattan — clearer skies, better image quality
s2_collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(amuwo_geom) \
    .filter(ee.Filter.calendarRange(11, 3, 'month')) \
    .filter(ee.Filter.date('2020-01-01', '2023-12-31')) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
    .map(mask_s2_clouds) \
    .map(compute_ndvi)

# Median composite reduces remaining noise and residual cloud artefacts
ndvi = s2_collection.select('ndvi') \
                    .median() \
                    .clip(amuwo_geom) \
                    .rename('ndvi')

# Verify NDVI statistics
ndvi_stats = ndvi.reduceRegion(
    reducer=ee.Reducer.minMax().combine(
        reducer2=ee.Reducer.mean(),
        sharedInputs=True
    ),
    geometry=amuwo_geom,
    scale=10,
    maxPixels=1e9
).getInfo()

print("✓ Sentinel-2 NDVI computed (dry season composite 2020-2023)")
print()
print("NDVI Statistics:")
print(f"  Minimum : {ndvi_stats['ndvi_min']:.4f}")
print(f"  Maximum : {ndvi_stats['ndvi_max']:.4f}")
print(f"  Mean    : {ndvi_stats['ndvi_mean']:.4f}")

✓ Sentinel-2 NDVI computed (dry season composite 2020-2023)

NDVI Statistics:
  Minimum : -0.2163
  Maximum : 0.7787
  Mean    : 0.2951


In [5]:
# ── CELL 5: Visualisation ────────────────────────────────────────────────────

amuwo_centroid = amuwo_odofin.geometry().centroid().coordinates().getInfo()
lon, lat = amuwo_centroid[0], amuwo_centroid[1]

Map = geemap.Map(center=[lat, lon], zoom=12)

# Study area boundary
amuwo_style = {'color': 'FF0000', 'fillColor': '00000000', 'width': 3}
Map.addLayer(amuwo_odofin.style(**amuwo_style), {}, 'Amuwo Odofin Boundary')

# Rainfall — light yellow to deep blue
Map.addLayer(annual_rainfall, {
    'min': 1200, 'max': 1800,
    'palette': ['FFFFE0', 'ADD8E6', '0000FF', '000033']
}, 'Mean Annual Rainfall (mm)')

# LULC — WorldCover standard colour scheme
Map.addLayer(lulc, {
    'min': 10, 'max': 100,
    'palette': ['006400', 'FFBB22', 'FFFF4C', 'F096FF',
                'FA0000', 'B4B4B4', 'F0F0F0', '0064C8',
                '0096A0', '00CF75', 'FAE6A0']
}, 'ESA WorldCover LULC 2021')

# NDVI — red (bare/built) to dark green (dense vegetation)
Map.addLayer(ndvi, {
    'min': -0.2, 'max': 0.8,
    'palette': ['FF0000', 'FFFF00', '00FF00', '006400']
}, 'NDVI (Sentinel-2 Dry Season Composite)')

Map.addLayerControl()
Map

Map(center=[6.455061708555176, 3.265078291524212], controls=(WidgetControl(options=['position', 'transparent_b…

In [6]:
# ── CELL 6: Stack and Export to Google Drive ─────────────────────────────────
# All bands cast to Float32 for type consistency across the feature stack
# Scale set to 100m as a compromise between CHIRPS (5km) and Sentinel-2 (10m)
# 100m is the standard resampling resolution for multi-source geospatial ML studies

rainfall_lulc_ndvi_stack = annual_rainfall.toFloat() \
    .addBands(rainy_days.toFloat()) \
    .addBands(extreme_rain.toFloat()) \
    .addBands(lulc.toFloat()) \
    .addBands(ndvi.toFloat())

print("Stacked band names:", rainfall_lulc_ndvi_stack.bandNames().getInfo())

export_task = ee.batch.Export.image.toDrive(
    image=rainfall_lulc_ndvi_stack,
    description='amuwo_odofin_rainfall_lulc_ndvi',
    folder='GeoAID_Project',
    fileNamePrefix='rainfall_lulc_ndvi_amuwo_odofin',
    region=amuwo_geom,
    scale=100,
    crs='EPSG:4326',
    maxPixels=1e9,
    fileFormat='GeoTIFF'
)

export_task.start()

print()
print("✓ Export task submitted to Google Drive")
print("  File  : rainfall_lulc_ndvi_amuwo_odofin.tif")
print("  Scale : 100 metres | Type: Float32")
print("  Bands : mean_annual_rainfall, mean_rainy_days_per_year,")
print("          extreme_rain_frequency, lulc, ndvi")
print()
print("Monitor at: https://code.earthengine.google.com/tasks")

Stacked band names: ['mean_annual_rainfall', 'mean_rainy_days_per_year', 'extreme_rain_frequency', 'lulc', 'ndvi']

✓ Export task submitted to Google Drive
  File  : rainfall_lulc_ndvi_amuwo_odofin.tif
  Scale : 100 metres | Type: Float32
  Bands : mean_annual_rainfall, mean_rainy_days_per_year,
          extreme_rain_frequency, lulc, ndvi

Monitor at: https://code.earthengine.google.com/tasks


In [7]:
# ── CELL 7: Session Summary ──────────────────────────────────────────────────
print("=" * 55)
print("  NOTEBOOK 03 — RAINFALL, LULC, NDVI: COMPLETE")
print("=" * 55)
print()
print("  Datasets acquired:")
print("  ├── CHIRPS Rainfall 2013-2023 (UCSB-CHG) ✓")
print("  ├── ESA WorldCover 2021 (10m LULC) ✓")
print("  └── Sentinel-2 NDVI dry season composite ✓")
print()
print("  Features added to matrix:")
print("  ├── Mean annual rainfall ✓")
print("  ├── Mean rainy days per year ✓")
print("  ├── Extreme rainfall frequency (>50mm/day) ✓")
print("  ├── Land use / land cover class ✓")
print("  └── NDVI (vegetation index) ✓")
print()
print("  Running total: 11 of 12 features complete")
print("  Remaining: Distance to drainage (NB04)")
print()
print("  Next: 04_soil_distance_features.ipynb")
print("        → Soil permeability + distance to rivers")
print("        → Complete the 12-feature matrix")
print("=" * 55)

  NOTEBOOK 03 — RAINFALL, LULC, NDVI: COMPLETE

  Datasets acquired:
  ├── CHIRPS Rainfall 2013-2023 (UCSB-CHG) ✓
  ├── ESA WorldCover 2021 (10m LULC) ✓
  └── Sentinel-2 NDVI dry season composite ✓

  Features added to matrix:
  ├── Mean annual rainfall ✓
  ├── Mean rainy days per year ✓
  ├── Extreme rainfall frequency (>50mm/day) ✓
  ├── Land use / land cover class ✓
  └── NDVI (vegetation index) ✓

  Running total: 11 of 12 features complete
  Remaining: Distance to drainage (NB04)

  Next: 04_soil_distance_features.ipynb
        → Soil permeability + distance to rivers
        → Complete the 12-feature matrix
